# CLARA en Google Colab (GPU)

Corre el pipeline de CLARA en **GPU** — mucho más rápido que el CPU local (donde un video de ~40 min son horas).

## Antes de empezar
1. **Runtime → Change runtime type → T4 GPU** (o mejor).
2. Sube a tu Google Drive, en una carpeta (p.ej. `MyDrive/clara/`):
   - el **video** `.mp4`
   - el **modelo de balón** `VballNetV1.onnx` (0.2 MB, no está en el repo)
3. Ejecuta las celdas **en orden**. Lo único que ajustas es la celda **CONFIG**.

> Calibración: el encuadre del video cambia a lo largo de los 40 min, así que **revisa el overlay QC** que genera la celda 7 antes de confiar en las zonas. Para un segmento de encuadre constante, una sola `cal.json` basta.

In [ ]:
# 1) GPU disponible?
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — cambia el runtime a GPU')

In [ ]:
# 2) Clonar (o actualizar) el repo CLARA
import os
REPO = '/content/CLARA'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/isaac09-ux/CLARA.git {REPO}
else:
    !cd {REPO} && git pull --ff-only
%cd {REPO}

In [ ]:
# 3) Dependencias. torch ya viene en Colab con CUDA; instalamos el resto.
# onnxruntime-gpu acelera VballNet en GPU (ball_vballnet usa CUDAExecutionProvider si está).
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1
!pip -q install ultralytics easyocr onnxruntime-gpu opencv-python-headless
import onnxruntime as ort
print('ORT providers:', ort.get_available_providers())

In [ ]:
# 4) Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 5) CONFIG — ajusta a tu Drive
import os
DRIVE = '/content/drive/MyDrive/clara'          # carpeta donde subiste el video y el modelo
VIDEO = f'{DRIVE}/20260529_195308.mp4'          # tu video
BALL_MODEL = f'{DRIVE}/VballNetV1.onnx'         # modelo de balón
OUT = '/content/out'                            # salida local (luego se copia a Drive)
HALF_COURT = True                               # media cancha 9x9
# Esquinas (px) media cancha, orden: cercana_izq, cercana_der, lejana_der, lejana_izq.
# (Estimadas para el encuadre ~08:00–20:00; verifica el QC de la celda 7.)
CORNERS = [(250, 930), (1680, 930), (1530, 690), (410, 690)]
assert os.path.exists(VIDEO), f'No existe el video: {VIDEO}'
assert os.path.exists(BALL_MODEL), f'No existe el modelo: {BALL_MODEL}'
print('config OK:', VIDEO)

In [ ]:
# 6) (OPCIONAL) Recortar un segmento y/o bajar fps para ir más rápido o probar.
# Descomenta para usar. -ss antes de -i = seek rápido; re-encode a 30 fps.
# SEG = '/content/seg.mp4'
# !ffmpeg -y -hide_banner -loglevel error -ss 00:08:00 -i "{VIDEO}" -t 120 -r 30 -an -c:v libx264 -crf 20 {SEG}
# VIDEO = SEG; print('usando segmento', VIDEO)

In [ ]:
# 7) Calibración SIN GUI: genera cal.json desde CORNERS + overlay QC para verificar.
import cv2, numpy as np, json
cap = cv2.VideoCapture(VIDEO); ok, frame = cap.read(); cap.release()
assert ok, 'no pude leer un frame del video'
Hh, Ww = frame.shape[:2]
src = np.float32(CORNERS); ppm = 40
court_h_m = 9 if HALF_COURT else 18
cw, ch = 9 * ppm, court_h_m * ppm
dst = (np.float32([[0,0],[cw,0],[cw,ch],[0,ch]]) if HALF_COURT
       else np.float32([[cw,0],[cw,ch],[0,ch],[0,0]]))
Hm, _ = cv2.findHomography(src, dst)
cal = {'video_reference': os.path.basename(VIDEO), 'frame_shape': [Hh, Ww],
       'court_size_m': [9, court_h_m], 'pixels_per_meter': ppm,
       'pixel_corners': src.astype(int).tolist(),
       'homography_matrix': Hm.tolist(), 'half_court': HALF_COURT}
CAL = '/content/cal.json'
open(CAL, 'w').write(json.dumps(cal, indent=2))
# QC overlay: proyecta el perímetro de la cancha de vuelta al frame
Hinv = np.linalg.inv(Hm)
def to_img(pts_m):
    pts = (np.array(pts_m, np.float32) * ppm).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, Hinv).reshape(-1, 2).astype(int)
peri = to_img([[0,0],[9,0],[9,court_h_m],[0,court_h_m]])
vis = frame.copy(); cv2.polylines(vis, [peri.reshape(-1,1,2)], True, (0,220,0), 3)
cv2.imwrite('/content/qc_cal.jpg', vis)
from IPython.display import Image, display; display(Image('/content/qc_cal.jpg'))
print('Verifica que el contorno verde caiga sobre la cancha; si no, ajusta CORNERS en la celda 5 y re-corre.')

In [ ]:
# 8) (ALTERNATIVA automatica) Auto-calibracion por segmentacion, sin marcar esquinas.
# Util porque el encuadre cambia. Descomenta para usarla en vez de CORNERS/CAL.
# !pip -q install gdown
# !gdown 1__zkTmGwZo2z0EgbJvC14I_3kOpgQx3o -O /content/court_weights.zip
# !cd /content && unzip -o court_weights.zip >/dev/null
# COURT_SEG = '/content/weights/court/weights/best.pt'   # ajusta si la ruta difiere
# print('seg model:', os.path.exists(COURT_SEG))

In [ ]:
# 9) Correr CLARA en GPU
!python src/clara.py "{VIDEO}" --calibration {CAL} --ball-detector vballnet --vballnet-model "{BALL_MODEL}" --out {OUT}
# Para auto-cal por segmentacion (celda 8): quita --calibration y agrega:  --court-seg-model {COURT_SEG}

In [ ]:
# 10) Reporte HTML del coach
!python src/clara_report.py {OUT}/scouting_data.json --topdown {OUT}/topdown.png

In [ ]:
# 11) Copiar resultados a Drive y mostrar overlays
import shutil
name = os.path.splitext(os.path.basename(VIDEO))[0]
DEST = f'{DRIVE}/out_{name}'
shutil.copytree(OUT, DEST, dirs_exist_ok=True)
print('Resultados en tu Drive:', DEST)
from IPython.display import Image, display
for f in ['diagnostic.png', 'topdown.png']:
    p = f'{OUT}/{f}'
    if os.path.exists(p):
        print(f); display(Image(p))